# HC-WCI Analysis Notebook — FINAL
## Wright State University | Amir Hasan Khan

### What This Notebook Produces

| Part | Analysis | Paper Section |
|---|---|---|
| **1** | ACS ML Framework: 4-model comparison, bootstrap CI, group cross-validation | Section 3 |
| **2** | ACS Feature Importance and Country Analysis | Section 3 |
| **3** | AHS Validation: Rp vs Late Payment, chi-square, logistic regression | Section 4 |
| **4** | SCF Credit Exclusion: descriptive, logistic regression, trend | Section 5 |
| **5** | All paper tables exported as CSV | Tables 1-8 |
| **6** | All publication figures exported as PNG + PDF | Figures 1-4 |

### Prerequisites — install before running
```
pip install statsmodels shap
```

### Run Order
Top to bottom. Every cell prints its own status.

In [1]:
# ================================================================
# CELL 0 — MASTER CONFIGURATION
# Edit BASE if your research folder moves. Nothing else needs editing.
# ================================================================

import os

BASE = r"C:\Users\Amirh\OneDrive - Wright State University\Research\HC_WCI_Research"

# ── Input files ───────────────────────────────────────────────
ACS_CSV = os.path.join(BASE, '04_Code', 'HCWCI_Master_Dataset_v2.csv')
AHS_CSV = os.path.join(BASE, '01_AHS',  'working', 'AHS_clean.csv')
SCF_CSV = os.path.join(BASE, '03_SCF',  'working', 'SCF_clean.csv')

# ── Output directories ────────────────────────────────────────
OUT_DIR     = os.path.join(BASE, '05_Analysis')
TABLES_DIR  = os.path.join(OUT_DIR, 'tables')
FIGURES_DIR = os.path.join(OUT_DIR, 'figures')
MODELS_DIR  = os.path.join(OUT_DIR, 'models')
for d in [TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Configuration loaded.')
print(f'Base exists : {os.path.isdir(BASE)}')
print(f'Output dir  : {OUT_DIR}')

Configuration loaded.
Base exists : True
Output dir  : C:\Users\Amirh\OneDrive - Wright State University\Research\HC_WCI_Research\05_Analysis


In [3]:
# ================================================================
# CELL 1 — IMPORTS AND HELPERS
# ================================================================

import sys, pickle, warnings
import numpy  as np
import pandas as pd
from datetime import datetime

from sklearn.ensemble        import RandomForestRegressor
from sklearn.linear_model    import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics         import r2_score, mean_squared_error
from sklearn.preprocessing   import StandardScaler
from scipy.stats             import chi2_contingency, mannwhitneyu

try:
    import statsmodels.api as sm
    HAS_SM = True
    print('statsmodels : available')
except ImportError:
    HAS_SM = False
    print('statsmodels : NOT found  -->  pip install statsmodels')

try:
    import shap
    HAS_SHAP = True
    print('shap        : available')
except ImportError:
    HAS_SHAP = False
    print('shap        : NOT found  -->  pip install shap')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
SEED = 42
np.random.seed(SEED)

print(f'\nPython {sys.version.split()[0]}  pandas {pd.__version__}  numpy {np.__version__}')
print(f'Started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

# ── Helpers ───────────────────────────────────────────────────
LOG = []

def log(msg, level='INFO'):
    ts   = datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {level:<8} {msg}'
    print(line); LOG.append(line)

def section(title):
    bar = '=' * 65
    msg = f'\n{bar}\n  {title}\n{bar}'
    print(msg); LOG.append(msg)

def save_table(df, fname, desc=''):
    path = os.path.join(TABLES_DIR, fname)
    df.to_csv(path, encoding='utf-8')
    log(f'Table saved : {fname}  {desc}')

def save_fig(fig, fname, dpi=300):
    for ext in ['.png', '.pdf']:
        fig.savefig(os.path.join(FIGURES_DIR, fname + ext),
                    dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    log(f'Figure saved: {fname}')

def bootstrap_r2_ci(y_true, y_pred, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    yt  = np.array(y_true)
    yp  = np.array(y_pred)
    n   = len(yt)
    r2s = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        r2s.append(r2_score(yt[idx], yp[idx]))
    return np.percentile(r2s, [2.5, 97.5])

log('Imports and helpers loaded.')

statsmodels : available
shap        : available

Python 3.12.7  pandas 2.2.2  numpy 1.26.4
Started: 2026-05-11 22:39:46
[22:39:46] INFO     Imports and helpers loaded.


In [5]:
# ================================================================
# CELL 2 — LOAD AND VALIDATE ALL DATA
# ================================================================

section('LOAD AND VALIDATE ALL DATA')

for label, path in [('ACS', ACS_CSV), ('AHS', AHS_CSV), ('SCF', SCF_CSV)]:
    ok   = os.path.isfile(path)
    size = f'{os.path.getsize(path)/1e6:.1f} MB' if ok else 'MISSING'
    mark = 'OK  ' if ok else 'FAIL'
    print(f'  [{mark}]  {label:4}  {size:10}  {path}')
    if not ok:
        raise FileNotFoundError(
            f'{label} file missing at: {path}\n'
            'Check Cell 0 paths and confirm the extraction notebook ran successfully.'
        )

acs = pd.read_csv(ACS_CSV)
ahs = pd.read_csv(AHS_CSV)
scf = pd.read_csv(SCF_CSV)

log(f'ACS: {len(acs):,} cohorts | AHS: {len(ahs):,} units | SCF: {len(scf):,} families')

# Assertions — hard stop if core variables are missing
assert 'Mortgaged_Ownership_Rate' in acs.columns, 'ACS target variable missing'
assert 'LATE_PAYMENT_FLAG'        in ahs.columns, 'AHS late payment flag missing'
assert 'CREDIT_EXCLUDED'          in scf.columns, 'SCF credit exclusion missing'
assert 'Rent_Performance_Ratio'   in acs.columns, 'ACS Rp variable missing'

log('All assertions passed. Safe to continue.')


  LOAD AND VALIDATE ALL DATA
  [OK  ]  ACS   5.0 MB      C:\Users\Amirh\OneDrive - Wright State University\Research\HC_WCI_Research\04_Code\HCWCI_Master_Dataset_v2.csv
  [OK  ]  AHS   7.3 MB      C:\Users\Amirh\OneDrive - Wright State University\Research\HC_WCI_Research\01_AHS\working\AHS_clean.csv
  [OK  ]  SCF   3.1 MB      C:\Users\Amirh\OneDrive - Wright State University\Research\HC_WCI_Research\03_SCF\working\SCF_clean.csv
[22:39:52] INFO     ACS: 30,397 cohorts | AHS: 55,669 units | SCF: 16,620 families
[22:39:52] INFO     All assertions passed. Safe to continue.


In [7]:
# ================================================================
# CELL 3 — ACS: PREPROCESSING
# Exact replication of AI_framework.ipynb (rectified version).
# All three fixes applied: cohort filter, Rp winsorise, NaN drop.
# ================================================================

section('PART 1 — ACS PREPROCESSING')

edu_map = {
    '1_Low (No/Primary)'              : 1,
    '2_Some High School'              : 2,
    '3_HS Graduate'                   : 3,
    '4_Some College'                  : 4,
    '5_College+ (Bachelor or Higher)' : 5,
}
acs['Edu_Tier'] = acs['Education_Group'].map(edu_map)

# FIX 1: remove degenerate tiny cohorts
n0  = len(acs)
acs = acs[acs['Total_People'] >= 100].copy()
log(f'Cohort filter: {n0:,} -> {len(acs):,}  (removed {n0 - len(acs):,})')

# FIX 2: winsorise Rp at 95th percentile
rp_cap      = acs['Rent_Performance_Ratio'].quantile(0.95)
acs['Rp_Win'] = acs['Rent_Performance_Ratio'].clip(upper=rp_cap)
log(f'Rp winsorized cap (95th pct): {rp_cap:.4f}')

# FIX 3: drop any residual NaN rows
TARGET = 'Mortgaged_Ownership_Rate'
FEATS  = ['Years_in_US', 'Edu_Tier', 'Rp_Win', 'Avg_Income']
acs    = acs.dropna(subset=FEATS + [TARGET]).copy()

log(f'Final ACS dataset: {len(acs):,} cohorts | Pop: {acs["Total_People"].sum():,.0f}')

print()
print('Cohorts by country:')
print(acs.groupby('Country').agg(
    N_cohorts  = ('Total_People', 'count'),
    Population = ('Total_People', 'sum'),
).assign(Population=lambda d: d['Population'].map('{:,.0f}'.format)).to_string())

print()
print('Target (Mortgaged_Ownership_Rate) summary:')
print(acs[TARGET].describe().round(3).to_string())


  PART 1 — ACS PREPROCESSING
[22:39:55] INFO     Cohort filter: 30,397 -> 30,397  (removed 0)
[22:39:55] INFO     Rp winsorized cap (95th pct): 1.7664
[22:39:55] INFO     Final ACS dataset: 30,397 cohorts | Pop: 166,261,329

Cohorts by country:
                    N_cohorts  Population
Country                                  
Bangladesh               1589   2,310,705
China                    3894  18,349,782
Dominican Republic       2605   9,270,198
India                    3646  22,864,410
Kenya                    1776   1,254,540
Mexico                   6359  85,224,364
Nigeria                  2191   3,355,968
Philippines              3808  14,277,467
Turkey                   1261     990,734
Vietnam                  3268   8,363,161

Target (Mortgaged_Ownership_Rate) summary:
count   30,397.0000
mean        32.2480
std         24.1760
min          0.0000
25%         11.5610
50%         31.0930
75%         50.3090
max         99.6380


In [9]:
# ================================================================
# CELL 4 — ACS: FOUR-MODEL ML FRAMEWORK
#
# Model A : HC-WCI RF  (Years + Edu + Rp,  population-weighted)
# Model B : Income RF  (same algorithm  -- fair comparison to A)
# Model C : Income LR  (original baseline)
# Model D : HC-WCI RF  (Years + Edu, NO Rp) -- HEADLINE RESULT
#
# FIX: sample weight = Total_People throughout
# FIX: R2 is Coefficient of Determination, NOT accuracy
# ================================================================

section('PART 1 — FOUR-MODEL ML FRAMEWORK')

y  = acs[TARGET]
sw = acs['Total_People']

X_A = acs[['Years_in_US', 'Edu_Tier', 'Rp_Win']]
X_B = acs[['Avg_Income']]
X_D = acs[['Years_in_US', 'Edu_Tier']]

# Identical seed for all splits
XA_tr, XA_te, y_tr, y_te, sw_tr, sw_te = train_test_split(
    X_A, y, sw, test_size=0.2, random_state=SEED)
XB_tr, XB_te, _, _ = train_test_split(X_B, y, test_size=0.2, random_state=SEED)
XD_tr, XD_te, _, _ = train_test_split(X_D, y, test_size=0.2, random_state=SEED)

log(f'Train: {len(y_tr):,}  |  Test: {len(y_te):,}')

RF = dict(n_estimators=200, random_state=SEED, n_jobs=-1)

print('Training Model A (HC-WCI RF with Rp, population-weighted)...')
rf_A = RandomForestRegressor(**RF)
rf_A.fit(XA_tr, y_tr, sample_weight=sw_tr)
pred_A = rf_A.predict(XA_te)

print('Training Model B (Income-only RF, fair baseline)...')
rf_B = RandomForestRegressor(**RF)
rf_B.fit(XB_tr, y_tr)
pred_B = rf_B.predict(XB_te)

print('Training Model C (Income-only Linear, original baseline)...')
lr_C = LinearRegression()
lr_C.fit(XB_tr, y_tr)
pred_C = lr_C.predict(XB_te)

print('Training Model D (HC-WCI RF Years+Edu, NO Rp, HEADLINE)...')
rf_D = RandomForestRegressor(**RF)
rf_D.fit(XD_tr, y_tr)
pred_D = rf_D.predict(XD_te)

print('Computing 1000-iteration bootstrap confidence intervals...')

results = []
for label, yp in [
    ('Model A: HC-WCI RF (Years+Edu+Rp, weighted)', pred_A),
    ('Model B: Income-only RF  (fair baseline)',     pred_B),
    ('Model C: Income-only Linear  (orig baseline)', pred_C),
    ('Model D: HC-WCI RF  (Years+Edu, NO Rp)',      pred_D),
]:
    r2   = r2_score(y_te, yp)
    rmse = np.sqrt(mean_squared_error(y_te, yp))
    lo, hi = bootstrap_r2_ci(y_te, yp)
    results.append({'Model': label,
                    'R2_pct': round(r2 * 100, 2),
                    'CI_lo' : round(lo * 100, 2),
                    'CI_hi' : round(hi * 100, 2),
                    'RMSE'  : round(rmse, 4)})

results_df = pd.DataFrame(results)

print()
print('=' * 75)
print('  MODEL PERFORMANCE — COEFFICIENT OF DETERMINATION (R2)')
print('  R2 = proportion of variance explained  NOT accuracy')
print('=' * 75)
for _, r in results_df.iterrows():
    print(f"  {r['Model']:50}  R2={r['R2_pct']:6.2f}%  "
          f"95%CI=[{r['CI_lo']:.2f}%,{r['CI_hi']:.2f}%]  "
          f"RMSE={r['RMSE']:.4f}")

r2_C = results_df.loc[2, 'R2_pct']
r2_D = results_df.loc[3, 'R2_pct']
print(f'\nKEY FINDING: Model D = {r2_D:.2f}%  vs  Model C = {r2_C:.2f}%')
print(f'HC-WCI (Years+Edu) explains {r2_D - r2_C:.2f} additional pp above income alone.')


  PART 1 — FOUR-MODEL ML FRAMEWORK
[22:39:59] INFO     Train: 24,317  |  Test: 6,080
Training Model A (HC-WCI RF with Rp, population-weighted)...
Training Model B (Income-only RF, fair baseline)...
Training Model C (Income-only Linear, original baseline)...
Training Model D (HC-WCI RF Years+Edu, NO Rp, HEADLINE)...
Computing 1000-iteration bootstrap confidence intervals...

  MODEL PERFORMANCE — COEFFICIENT OF DETERMINATION (R2)
  R2 = proportion of variance explained  NOT accuracy
  Model A: HC-WCI RF (Years+Edu+Rp, weighted)         R2= -3.53%  95%CI=[-6.44%,-0.60%]  RMSE=24.6855
  Model B: Income-only RF  (fair baseline)            R2=-22.69%  95%CI=[-26.70%,-19.31%]  RMSE=26.8723
  Model C: Income-only Linear  (orig baseline)        R2=  5.59%  95%CI=[3.94%,6.94%]  RMSE=23.5732
  Model D: HC-WCI RF  (Years+Edu, NO Rp)              R2= 14.02%  95%CI=[12.30%,15.62%]  RMSE=22.4959

KEY FINDING: Model D = 14.02%  vs  Model C = 5.59%
HC-WCI (Years+Edu) explains 8.43 additional pp above

In [27]:
# ================================================================
# CELL 5 — ACS: LEAVE-ONE-COUNTRY-OUT CROSS-VALIDATION
# With 10 countries and 10 folds this equals LOCO-CV.
# Each fold tests on one held-out country entirely.
# ================================================================

section('ACS — LEAVE-ONE-COUNTRY-OUT CROSS-VALIDATION')

print('NOTE: 10 countries + 10 GroupKFold = Leave-One-Country-Out (LOCO-CV)')
print('Each fold trains on 9 countries and tests on the 10th.')
print('This is stricter than random k-fold and tests cross-community generalization.')
print()

groups = acs['Country'].values
gkf    = GroupKFold(n_splits=10)

cv_configs = [
    ('Model D: HC-WCI (Years+Edu, no Rp)', X_D, 'rf',  False),
    ('Model B: Income-only RF',             X_B, 'rf',  False),
    ('Model C: Income-only Linear',         X_B, 'lr',  False),
    ('Model A: HC-WCI (Years+Edu+Rp)',      X_A, 'rf',  True),
]

cv_rows = []
fold_detail_rows = []

for name, X_cv, mtype, weighted in cv_configs:
    fold_r2     = []
    fold_labels = []
    for fold_num, (tr_idx, te_idx) in enumerate(gkf.split(X_cv, y, groups=groups)):
        Xtr, Xte  = X_cv.iloc[tr_idx], X_cv.iloc[te_idx]
        ytr, yte  = y.iloc[tr_idx],    y.iloc[te_idx]
        swtr      = sw.iloc[tr_idx]

        # Which country is being held out in this fold?
        held_out = acs.iloc[te_idx]['Country'].iloc[0]

        if mtype == 'rf':
            m = RandomForestRegressor(**RF)
            m.fit(Xtr, ytr, sample_weight=swtr if weighted else None)
        else:
            m = LinearRegression()
            m.fit(Xtr, ytr)

        r2_fold = r2_score(yte, m.predict(Xte))
        fold_r2.append(r2_fold)
        fold_labels.append(held_out)

        if 'Model D' in name:
            fold_detail_rows.append({
                'Held_Out_Country' : held_out,
                'R2_pct'           : round(r2_fold * 100, 2),
            })

    cv_rows.append({
        'Model'                 : name,
        'LOCO_CV_R2_Mean_pct'   : round(np.mean(fold_r2) * 100, 2),
        'LOCO_CV_R2_Std_pct'    : round(np.std(fold_r2)  * 100, 2),
        'LOCO_CV_R2_Min_pct'    : round(np.min(fold_r2)  * 100, 2),
        'LOCO_CV_R2_Max_pct'    : round(np.max(fold_r2)  * 100, 2),
    })
    print(f"  {name:48}  "
          f"mean={cv_rows[-1]['LOCO_CV_R2_Mean_pct']:.2f}%  "
          f"std={cv_rows[-1]['LOCO_CV_R2_Std_pct']:.2f}%")

cv_df = pd.DataFrame(cv_rows)
print()
print(cv_df.to_string(index=False))

# Model D fold detail (shows which countries the model struggles with)
print()
print('Model D performance when each country is the test set:')
fold_detail_df = pd.DataFrame(fold_detail_rows).sort_values('R2_pct', ascending=False)
print(fold_detail_df.to_string(index=False))


  ACS — LEAVE-ONE-COUNTRY-OUT CROSS-VALIDATION
NOTE: 10 countries + 10 GroupKFold = Leave-One-Country-Out (LOCO-CV)
Each fold trains on 9 countries and tests on the 10th.
This is stricter than random k-fold and tests cross-community generalization.

  Model D: HC-WCI (Years+Edu, no Rp)                mean=3.43%  std=10.43%
  Model B: Income-only RF                           mean=-30.99%  std=20.85%
  Model C: Income-only Linear                       mean=-3.74%  std=9.00%
  Model A: HC-WCI (Years+Edu+Rp)                    mean=-18.61%  std=12.69%

                             Model  LOCO_CV_R2_Mean_pct  LOCO_CV_R2_Std_pct  LOCO_CV_R2_Min_pct  LOCO_CV_R2_Max_pct
Model D: HC-WCI (Years+Edu, no Rp)               3.4300             10.4300            -16.5100             16.6800
           Model B: Income-only RF             -30.9900             20.8500            -78.1800            -11.4600
       Model C: Income-only Linear              -3.7400              9.0000            -25.1500 

In [13]:
# ================================================================
# CELL 6 — ACS: FEATURE IMPORTANCE + COUNTRY ANALYSIS
# ================================================================

section('ACS — FEATURE IMPORTANCE AND COUNTRY ANALYSIS')

fi_A = pd.Series(rf_A.feature_importances_,
                  index=['Years_in_US', 'Edu_Tier', 'Rp_Win'])
fi_D = pd.Series(rf_D.feature_importances_,
                  index=['Years_in_US', 'Edu_Tier'])

print('Feature Importance (Gini) — Model A (with Rp):')
for f, v in fi_A.items():
    print(f'  {f:20}: {v*100:.2f}%')

print()
print('Feature Importance (Gini) — Model D (without Rp, headline):')
for f, v in fi_D.items():
    print(f'  {f:20}: {v*100:.2f}%')

# Bootstrap CI on Model D feature importance (200 iterations)
print()
print('Bootstrap CI on Model D feature importance (200 iterations)...')
rng_fi = np.random.default_rng(SEED)
fi_boot = {'Years_in_US': [], 'Edu_Tier': []}
for _ in range(200):
    idx = rng_fi.integers(0, len(XD_tr), size=len(XD_tr))
    mb  = RandomForestRegressor(n_estimators=50,
                                 random_state=int(rng_fi.integers(9999)),
                                 n_jobs=-1)
    mb.fit(XD_tr.iloc[idx], y_tr.iloc[idx])
    for i, nm in enumerate(['Years_in_US', 'Edu_Tier']):
        fi_boot[nm].append(mb.feature_importances_[i])

fi_ci_rows = []
for nm in ['Years_in_US', 'Edu_Tier']:
    v = np.array(fi_boot[nm])
    fi_ci_rows.append({
        'Feature'   : nm,
        'Importance': round(fi_D[nm] * 100, 2),
        'CI_lo'     : round(np.percentile(v, 2.5)  * 100, 2),
        'CI_hi'     : round(np.percentile(v, 97.5) * 100, 2),
    })
fi_ci_df = pd.DataFrame(fi_ci_rows)
print(fi_ci_df.to_string(index=False))

# Country-level R2
print()
print('Model D R2 by country of origin:')
test_idx = XD_te.index
c_rows   = []
for c in sorted(acs['Country'].unique()):
    mask = acs.loc[test_idx, 'Country'] == c
    if mask.sum() < 15:
        continue
    r2c = r2_score(y_te[mask], pred_D[mask.values])
    c_rows.append({'Country': c,
                   'N_test_cohorts': int(mask.sum()),
                   'R2_pct': round(r2c * 100, 2)})
country_df = pd.DataFrame(c_rows).sort_values('R2_pct', ascending=False)
print(country_df.to_string(index=False))


  ACS — FEATURE IMPORTANCE AND COUNTRY ANALYSIS
Feature Importance (Gini) — Model A (with Rp):
  Years_in_US         : 35.88%
  Edu_Tier            : 20.85%
  Rp_Win              : 43.27%

Feature Importance (Gini) — Model D (without Rp, headline):
  Years_in_US         : 63.00%
  Edu_Tier            : 37.00%

Bootstrap CI on Model D feature importance (200 iterations)...
    Feature  Importance   CI_lo   CI_hi
Years_in_US     63.0000 60.0500 66.0700
   Edu_Tier     37.0000 33.9300 39.9500

Model D R2 by country of origin:
           Country  N_test_cohorts   R2_pct
            Turkey             260  17.6700
             China             780  16.0600
             India             729  15.5300
           Nigeria             415  14.9800
        Bangladesh             296  13.3900
           Vietnam             653  10.9500
            Mexico            1262   8.5600
             Kenya             373   0.0300
Dominican Republic             525  -2.6500
       Philippines            

In [15]:
# ================================================================
# CELL 7 — AHS: DESCRIPTIVE STATISTICS
# ================================================================

section('PART 2 — AHS DESCRIPTIVE STATISTICS')

DIVISION_NAMES = {
    1: 'New England',      2: 'Middle Atlantic',    3: 'E North Central',
    4: 'W North Central',  5: 'South Atlantic',     6: 'E South Central',
    7: 'W South Central',  8: 'Mountain',           9: 'Pacific',
}

ahs_r = ahs[ahs['IS_RENTER'] == 1].copy()
ahs_o = ahs[ahs['IS_OWNER']  == 1].copy()

print(f'AHS 2023 National PUF')
print(f'  Total housing units : {len(ahs):,}')
print(f'  Owners              : {len(ahs_o):,} ({len(ahs_o)/len(ahs)*100:.1f}%)')
print(f'  Renters             : {len(ahs_r):,} ({len(ahs_r)/len(ahs)*100:.1f}%)')
print(f'  No-cash-rent        : {(ahs["IS_OWNER"]+ahs["IS_RENTER"]==0).sum():,}')

desc_rows = []
for var in ['EDU_TIER', 'INCOME', 'Rp', 'YEARS_US', 'HC_WCI']:
    for grp, dg in [('All', ahs), ('Renters', ahs_r), ('Owners', ahs_o)]:
        v = dg[var].dropna()
        if len(v) == 0:
            continue
        desc_rows.append({'Variable': var, 'Group': grp, 'N': len(v),
                           'Mean': round(v.mean(), 3), 'Median': round(v.median(), 3),
                           'SD': round(v.std(), 3)})
desc_df = pd.DataFrame(desc_rows)
print()
print('Descriptive statistics by tenure:')
print(desc_df.to_string(index=False))

print()
ahs_lp = ahs[ahs['LATE_PAYMENT_FLAG'].notna()]
print(f'Housing Insecurity module: {len(ahs_lp):,} units ({len(ahs_lp)/len(ahs)*100:.1f}%)')
print(f'  Behind payments : {ahs_lp["LATE_PAYMENT_FLAG"].eq(1).sum():,}'
      f' ({ahs_lp["LATE_PAYMENT_FLAG"].mean()*100:.2f}%)')
print(f'  Never behind    : {ahs_lp["LATE_PAYMENT_FLAG"].eq(0).sum():,}')
print(f'  Eviction threat : {ahs["EVICTION_THREAT"].eq(1).sum():,}')

print()
print('Median rent by Census Division (renters only):')
div_rent = (ahs_r.groupby('DIVISION')['RENT'].median()
            .reset_index()
            .assign(Division=lambda d: d['DIVISION'].map(DIVISION_NAMES)))
print(div_rent.to_string(index=False))


  PART 2 — AHS DESCRIPTIVE STATISTICS
AHS 2023 National PUF
  Total housing units : 55,669
  Owners              : 28,192 (50.6%)
  Renters             : 19,735 (35.5%)
  No-cash-rent        : 7,742

Descriptive statistics by tenure:
Variable   Group     N         Mean      Median           SD
EDU_TIER     All 48527       3.8740      4.0000       1.1150
EDU_TIER Renters 19735       3.5620      4.0000       1.1580
EDU_TIER  Owners 28192       4.0990      4.0000       1.0260
  INCOME     All 48527 103,685.4030 65,000.0000 148,705.7770
  INCOME Renters 19735  60,886.2150 37,000.0000 101,131.7200
  INCOME  Owners 28192 134,696.3810 90,875.0000 168,831.6010
      Rp     All 19735       1.0790      1.0000       0.6980
      Rp Renters 19735       1.0790      1.0000       0.6980
YEARS_US     All 48527      10.9870      7.0000      10.0170
YEARS_US Renters 19735       6.1260      4.0000       6.8650
YEARS_US  Owners 28192      14.3890     12.0000      10.4550
  HC_WCI     All 19735       0.64

In [17]:
# ================================================================
# CELL 8 — AHS: Rp VALIDATION
# THE TABLE THAT ANSWERS THE JHE EDITOR CRITIQUE.
#
# Tests:
#   1. Rp quartile vs LATE_PAYMENT_FLAG (descriptive table)
#   2. Chi-square test of independence
#   3. Mann-Whitney U — Q1 vs Q4
#   4. Logistic regression with controls
#   5. Robustness with HOUSING_DISTRESS outcome
# ================================================================

section('PART 2 — AHS Rp VALIDATION')

ahs_val = ahs[
    (ahs['IS_RENTER'] == 1) &
     ahs['LATE_PAYMENT_FLAG'].notna() &
     ahs['Rp'].notna()
].copy()
log(f'Validation subset: {len(ahs_val):,} renter observations')

ahs_val['Rp_Q'] = pd.qcut(
    ahs_val['Rp'], q=4,
    labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
)

q_tbl = (
    ahs_val.groupby('Rp_Q')['LATE_PAYMENT_FLAG']
    .agg(N='count', Late_Rate='mean')
    .assign(Late_Pct   = lambda d: (d['Late_Rate'] * 100).round(2),
            OnTime_Pct = lambda d: ((1 - d['Late_Rate']) * 100).round(2))
)

print('RP QUARTILE vs LATE PAYMENT RATE:')
print(q_tbl.to_string())

q1_rate   = q_tbl.loc['Q1 (Lowest)',  'Late_Pct']
q4_rate   = q_tbl.loc['Q4 (Highest)', 'Late_Pct']
reduction = (q1_rate - q4_rate) / q1_rate * 100
print(f'\nQ1 = {q1_rate:.2f}%  |  Q4 = {q4_rate:.2f}%  |  Risk reduction: {reduction:.1f}%')

# Chi-square
ct = pd.crosstab(ahs_val['Rp_Q'], ahs_val['LATE_PAYMENT_FLAG'])
chi2_val, p_chi2, dof, _ = chi2_contingency(ct)
print(f'\nChi-square: chi2={chi2_val:.4f}  df={dof}  p={p_chi2:.6f}')
print(f'Result: {"SIGNIFICANT (p<0.05)" if p_chi2 < 0.05 else "NOT significant"}')

# Mann-Whitney U
q1v = ahs_val[ahs_val['Rp_Q'] == 'Q1 (Lowest)']['LATE_PAYMENT_FLAG'].dropna()
q4v = ahs_val[ahs_val['Rp_Q'] == 'Q4 (Highest)']['LATE_PAYMENT_FLAG'].dropna()
mwu_s, mwu_p = mannwhitneyu(q1v, q4v, alternative='greater')
print(f'Mann-Whitney U (Q1 > Q4): stat={mwu_s:.2f}  p={mwu_p:.6f}')

# Logistic regression
print()
print('LOGISTIC REGRESSION: LATE_PAYMENT_FLAG ~ Rp + EDU_TIER + YEARS_US + log(INCOME) + DIVISION')

ahs_reg = ahs_val[['LATE_PAYMENT_FLAG','Rp','EDU_TIER','YEARS_US','INCOME','DIVISION']].dropna().copy()
ahs_reg['log_INC'] = np.log1p(ahs_reg['INCOME'].clip(lower=1))
div_d = pd.get_dummies(ahs_reg['DIVISION'].astype(int), prefix='D', drop_first=True)
X_lr  = pd.concat([ahs_reg[['Rp', 'EDU_TIER', 'YEARS_US', 'log_INC']], div_d],
                   axis=1).astype(float)
y_lr  = ahs_reg['LATE_PAYMENT_FLAG'].astype(float)

if HAS_SM:
    res_lr = sm.Logit(y_lr, sm.add_constant(X_lr)).fit(disp=False, maxiter=200)
    key    = ['const', 'Rp', 'EDU_TIER', 'YEARS_US', 'log_INC']
    ci_lr  = res_lr.conf_int()
    coef_a = pd.DataFrame({
        'Coef'  : res_lr.params[key].round(4),
        'SE'    : res_lr.bse[key].round(4),
        'p'     : res_lr.pvalues[key].round(6),
        'OR'    : np.exp(res_lr.params[key]).round(4),
        'OR_lo' : np.exp(ci_lr[0][key]).round(4),
        'OR_hi' : np.exp(ci_lr[1][key]).round(4),
    })
    print(coef_a.to_string())
    print(f'\nRp:  OR={np.exp(res_lr.params["Rp"]):.4f}  p={res_lr.pvalues["Rp"]:.6f}')
    print(f'Pseudo-R2 (McFadden): {res_lr.prsquared:.4f}  N={int(res_lr.nobs):,}')
else:
    Xsc = StandardScaler().fit_transform(X_lr)
    lrm = LogisticRegression(max_iter=1000, random_state=SEED)
    lrm.fit(Xsc, y_lr)
    print('(Install statsmodels for p-values and odds ratios.)')

# Robustness: HOUSING_DISTRESS
print()
print('ROBUSTNESS CHECK: Rp quartile vs HOUSING_DISTRESS (late OR eviction threat):')
ahs_dist = ahs[(ahs['IS_RENTER']==1) & ahs['HOUSING_DISTRESS'].notna() & ahs['Rp'].notna()].copy()
ahs_dist['Rp_Q'] = pd.qcut(ahs_dist['Rp'], q=4, labels=['Q1','Q2','Q3','Q4'])
rob_tbl = (
    ahs_dist.groupby('Rp_Q')['HOUSING_DISTRESS']
    .agg(N='count', Rate='mean')
    .assign(Rate_Pct=lambda d: (d['Rate'] * 100).round(2))
)
print(rob_tbl.to_string())
chi2_r, p_r, _, _ = chi2_contingency(
    pd.crosstab(ahs_dist['Rp_Q'], ahs_dist['HOUSING_DISTRESS']))
print(f'Chi-square: {chi2_r:.4f}  p={p_r:.6f}')


  PART 2 — AHS Rp VALIDATION
[22:42:07] INFO     Validation subset: 9,541 renter observations
RP QUARTILE vs LATE PAYMENT RATE:
                 N  Late_Rate  Late_Pct  OnTime_Pct
Rp_Q                                               
Q1 (Lowest)   2397     0.0864    8.6400     91.3600
Q2            2603     0.1030   10.3000     89.7000
Q3            2259     0.0978    9.7800     90.2200
Q4 (Highest)  2282     0.0465    4.6500     95.3500

Q1 = 8.64%  |  Q4 = 4.65%  |  Risk reduction: 46.2%

Chi-square: chi2=59.7268  df=3  p=0.000000
Result: SIGNIFICANT (p<0.05)
Mann-Whitney U (Q1 > Q4): stat=2844123.00  p=0.000000

LOGISTIC REGRESSION: LATE_PAYMENT_FLAG ~ Rp + EDU_TIER + YEARS_US + log(INCOME) + DIVISION
            Coef     SE      p     OR  OR_lo  OR_hi
const    -0.7849 0.2234 0.0004 0.4562 0.2944 0.7068
Rp       -0.1634 0.0608 0.0072 0.8492 0.7538 0.9566
EDU_TIER -0.2068 0.0332 0.0000 0.8132 0.7619 0.8679
YEARS_US  0.0002 0.0054 0.9663 1.0002 0.9896 1.0110
log_INC  -0.0504 0.0167 0.0

In [19]:
# ================================================================
# CELL 9 — SCF: CREDIT EXCLUSION ANALYSIS
# ================================================================

section('PART 3 — SCF CREDIT EXCLUSION ANALYSIS')

edu_labels = {2: 'Some HS', 3: 'HS Grad', 4: 'Some College', 5: 'Bachelor+'}
inc_labels = {1:'0-20%', 2:'20-40%', 3:'40-60%',
               4:'60-80%', 5:'80-90%', 6:'90-100%'}

print(f'SCF: {len(scf):,} families  |  2016/2019/2022  |  implicate 1 only')
print(f'  Owners    : {scf["IS_OWNER"].mean()*100:.1f}%')
print(f'  Renters   : {scf["IS_RENTER"].mean()*100:.1f}%')
print(f'  Excluded  : {scf["CREDIT_EXCLUDED"].mean()*100:.1f}%')

ce_edu = (
    scf[scf['CREDIT_EXCLUDED'].notna()]
    .groupby('EDU_TIER')['CREDIT_EXCLUDED']
    .agg(N='count', Rate='mean')
    .assign(Rate_Pct = lambda d: (d['Rate'] * 100).round(2),
            Label    = lambda d: d.index.map(edu_labels))
)
print()
print('Credit exclusion by education tier:')
print(ce_edu.to_string())

ce_inc = (
    scf[scf['CREDIT_EXCLUDED'].notna()]
    .groupby('inccat')['CREDIT_EXCLUDED']
    .agg(N='count', Rate='mean')
    .assign(Rate_Pct = lambda d: (d['Rate'] * 100).round(2),
            Label    = lambda d: d.index.map(inc_labels))
)
print()
print('Credit exclusion by income quintile:')
print(ce_inc.to_string())

ct_ce = pd.crosstab(scf['EDU_TIER'].dropna(), scf['CREDIT_EXCLUDED'].dropna())
chi2_ce, p_ce, dof_ce, _ = chi2_contingency(ct_ce)
print(f'\nChi-square (exclusion vs education): chi2={chi2_ce:.4f}  p={p_ce:.6f}  df={dof_ce}')

# Logistic regression
print()
print('LOGISTIC REGRESSION: CREDIT_EXCLUDED ~ EDU_TIER + log(INCOME) + IS_RENTER + age + married')
scf_reg = scf[['CREDIT_EXCLUDED','EDU_TIER','INCOME','IS_RENTER','age','married']].dropna().copy()
scf_reg['log_INC'] = np.log1p(scf_reg['INCOME'].clip(lower=1))
X_scf = scf_reg[['EDU_TIER','log_INC','IS_RENTER','age','married']].astype(float)
y_scf = scf_reg['CREDIT_EXCLUDED'].astype(float)

if HAS_SM:
    res_s  = sm.Logit(y_scf, sm.add_constant(X_scf)).fit(disp=False, maxiter=200)
    key_s  = ['const', 'EDU_TIER', 'log_INC', 'IS_RENTER', 'age', 'married']
    ci_s   = res_s.conf_int()
    coef_s = pd.DataFrame({
        'Coef'  : res_s.params[key_s].round(4),
        'SE'    : res_s.bse[key_s].round(4),
        'p'     : res_s.pvalues[key_s].round(6),
        'OR'    : np.exp(res_s.params[key_s]).round(4),
        'OR_lo' : np.exp(ci_s[0][key_s]).round(4),
        'OR_hi' : np.exp(ci_s[1][key_s]).round(4),
    })
    print(coef_s.to_string())
    print(f'\nEDU_TIER: OR={np.exp(res_s.params["EDU_TIER"]):.4f}  p={res_s.pvalues["EDU_TIER"]:.6f}')
    print(f'Pseudo-R2: {res_s.prsquared:.4f}  N={int(res_s.nobs):,}')
else:
    Xsc2 = StandardScaler().fit_transform(X_scf)
    lrs  = LogisticRegression(max_iter=1000, random_state=SEED)
    lrs.fit(Xsc2, y_scf)
    print('(Install statsmodels for p-values and odds ratios.)')

# Trend 2016-2022
print()
print('Credit exclusion trend 2016-2019-2022:')
trend = (
    scf[scf['CREDIT_EXCLUDED'].notna()]
    .groupby('SURVEY_YEAR')['CREDIT_EXCLUDED']
    .agg(N='count', Rate='mean')
    .assign(Rate_Pct=lambda d: (d['Rate'] * 100).round(2))
)
print(trend.to_string())


  PART 3 — SCF CREDIT EXCLUSION ANALYSIS
SCF: 16,620 families  |  2016/2019/2022  |  implicate 1 only
  Owners    : 67.5%
  Renters   : 32.5%
  Excluded  : 17.1%

Credit exclusion by education tier:
             N   Rate  Rate_Pct         Label
EDU_TIER                                     
2         1593 0.2963   29.6300       Some HS
3         3474 0.2444   24.4400       HS Grad
4         3931 0.2203   22.0300  Some College
5         7622 0.0853    8.5300     Bachelor+

Credit exclusion by income quintile:
           N   Rate  Rate_Pct    Label
inccat                                
1       2773 0.3112   31.1200    0-20%
2       2641 0.2760   27.6000   20-40%
3       2638 0.2305   23.0500   40-60%
4       2780 0.1363   13.6300   60-80%
5       1635 0.0746    7.4600   80-90%
6       4153 0.0327    3.2700  90-100%

Chi-square (exclusion vs education): chi2=771.9531  p=0.000000  df=3

LOGISTIC REGRESSION: CREDIT_EXCLUDED ~ EDU_TIER + log(INCOME) + IS_RENTER + age + married
             

In [21]:
# ================================================================
# CELL 10 — GENERATE ALL PAPER TABLES
# ================================================================

section('GENERATE PAPER TABLES')

# Table 1: ACS descriptive by country
t1 = acs.groupby('Country').agg(
    N_Cohorts       = ('Total_People', 'count'),
    Weighted_Pop    = ('Total_People', 'sum'),
    Mean_MortRate   = ('Mortgaged_Ownership_Rate', 'mean'),
    Median_MortRate = ('Mortgaged_Ownership_Rate', 'median'),
    Mean_Income     = ('Avg_Income', 'mean'),
    Mean_YearsUS    = ('Years_in_US', 'mean'),
    Mean_Rp         = ('Rp_Win', 'mean'),
).round(2)
save_table(t1, 'Table1_ACS_Descriptive_by_Country.csv')

# Table 2: Model performance
t2 = results_df.copy()
t2.columns = ['Model','R2 (%)','CI Low (%)','CI High (%)','RMSE']
save_table(t2, 'Table2_ML_Model_Performance.csv')

# Table 3: Feature importance
save_table(fi_ci_df, 'Table3_Feature_Importance.csv')

# Table 4: AHS Rp validation
save_table(q_tbl.assign(Chi2=chi2_val, Chi2_p=p_chi2, MWU_p=mwu_p),
           'Table4_AHS_Rp_Validation.csv')
if HAS_SM:
    save_table(coef_a, 'Table4b_AHS_Logistic_Regression.csv')

# Table 5: SCF credit exclusion
save_table(ce_edu, 'Table5a_SCF_CE_by_Education.csv')
save_table(ce_inc, 'Table5b_SCF_CE_by_Income.csv')
if HAS_SM:
    save_table(coef_s, 'Table5c_SCF_Logistic_Regression.csv')

# Table 6: Cross-validation
save_table(cv_df, 'Table6_CrossValidation.csv')

# Table 7: Country-level R2
save_table(country_df, 'Table7_Country_Performance.csv')

# Table 8: AHS descriptive
save_table(desc_df, 'Table8_AHS_Descriptive.csv')

print(f'\nAll 8 tables saved to: {TABLES_DIR}')


  GENERATE PAPER TABLES
[22:42:17] INFO     Table saved : Table1_ACS_Descriptive_by_Country.csv  
[22:42:17] INFO     Table saved : Table2_ML_Model_Performance.csv  
[22:42:17] INFO     Table saved : Table3_Feature_Importance.csv  
[22:42:17] INFO     Table saved : Table4_AHS_Rp_Validation.csv  
[22:42:17] INFO     Table saved : Table4b_AHS_Logistic_Regression.csv  
[22:42:17] INFO     Table saved : Table5a_SCF_CE_by_Education.csv  
[22:42:17] INFO     Table saved : Table5b_SCF_CE_by_Income.csv  
[22:42:17] INFO     Table saved : Table5c_SCF_Logistic_Regression.csv  
[22:42:17] INFO     Table saved : Table6_CrossValidation.csv  
[22:42:17] INFO     Table saved : Table7_Country_Performance.csv  
[22:42:17] INFO     Table saved : Table8_AHS_Descriptive.csv  

All 8 tables saved to: C:\Users\Amirh\OneDrive - Wright State University\Research\HC_WCI_Research\05_Analysis\tables


In [23]:
# ================================================================
# CELL 11 — GENERATE ALL PUBLICATION FIGURES
# PNG (300 DPI) and PDF for journal submission.
# ================================================================

section('GENERATE PUBLICATION FIGURES')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 12,   'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})
TEAL = '#1a5276'
GREY = '#85929e'

# Figure 1: 4-model performance comparison
fig1, ax1 = plt.subplots(figsize=(9, 5))
labs   = ['A\nHC-WCI RF\n(+Rp)', 'B\nIncome\nRF',
          'C\nIncome\nLinear', 'D\nHC-WCI RF\n(no Rp)']
r2v    = results_df['R2_pct'].tolist()
elo    = [r2v[i] - results_df.loc[i,'CI_lo'] for i in range(4)]
ehi    = [results_df.loc[i,'CI_hi'] - r2v[i] for i in range(4)]
bars1  = ax1.bar(labs, r2v, color=[GREY,GREY,GREY,TEAL],
                  yerr=[elo,ehi], capsize=5,
                  error_kw={'elinewidth':1.5}, edgecolor='white', width=0.5)
for b, v in zip(bars1, r2v):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
             f'{v:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax1.set_ylabel('R\u00b2 — Variance Explained (%)')
ax1.set_title('Figure 1: HC-WCI vs Income-Only Baselines (95% Bootstrap CI)', fontsize=11)
ax1.set_ylim(min(r2v)-5, max(r2v)+4)
plt.tight_layout()
save_fig(fig1, 'Figure1_Model_Performance')

# Figure 2: Rp quartile vs late payment
fig2, ax2 = plt.subplots(figsize=(8, 5))
qlabs = q_tbl.index.tolist()
lates = q_tbl['Late_Pct'].values
ns    = q_tbl['N'].values
bars2 = ax2.bar(qlabs, lates, color=[GREY,GREY,GREY,TEAL], edgecolor='white', width=0.5)
for b, v, n in zip(bars2, lates, ns):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
             f'{v:.2f}%\n(n={n:,})', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Late Payment Rate (%)')
ax2.set_xlabel('Rent Performance Ratio (Rp) Quartile')
ax2.set_title(
    f'Figure 2: Rp Quartile and Housing Payment Reliability\n'
    f'AHS 2023 Renters (\u03c7\u00b2={chi2_val:.2f}, p={p_chi2:.4f})',
    fontsize=11)
ax2.annotate(
    f'{reduction:.0f}% lower risk\n(Q4 vs Q1)',
    xy=(3, lates[3]), xytext=(2, lates[0]-1.0),
    arrowprops=dict(arrowstyle='->', lw=1.2), fontsize=9)
plt.tight_layout()
save_fig(fig2, 'Figure2_Rp_Validation')

# Figure 3: SCF credit exclusion (dual panel)
fig3, axes3 = plt.subplots(1, 2, figsize=(12, 5))
elabs = list(edu_labels.values())
evals = ce_edu['Rate_Pct'].values
b3a   = axes3[0].bar(elabs, evals, color=TEAL, edgecolor='white', width=0.5)
for b, v in zip(b3a, evals):
    axes3[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                  f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
axes3[0].set_title('Panel A: By Education Tier', fontsize=11)
axes3[0].set_ylabel('Credit Exclusion Rate (%)')
axes3[0].set_ylim(0, max(evals)*1.3)
axes3[0].tick_params(axis='x', labelrotation=10)

ilabs = ['Bot\n0-20%','20-40%','40-60%','60-80%','80-90%','Top\n90-100%']
ivals = ce_inc['Rate_Pct'].values
b3b   = axes3[1].bar(ilabs, ivals, color=TEAL, edgecolor='white', width=0.5)
for b, v in zip(b3b, ivals):
    axes3[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                  f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
axes3[1].set_title('Panel B: By Income Quintile', fontsize=11)
axes3[1].set_ylabel('Credit Exclusion Rate (%)')
axes3[1].set_ylim(0, max(ivals)*1.3)
fig3.suptitle(
    'Figure 3: Credit Market Exclusion Falls with Human Capital\n'
    'Survey of Consumer Finances 2016-2022 (n=16,620)',
    fontsize=12, y=1.02)
plt.tight_layout()
save_fig(fig3, 'Figure3_SCF_Credit_Exclusion')

# Figure 4: Feature importance
fig4, ax4 = plt.subplots(figsize=(7, 4))
fnames = fi_ci_df['Feature'].values
fvals  = fi_ci_df['Importance'].values
f_elo  = fvals - fi_ci_df['CI_lo'].values
f_ehi  = fi_ci_df['CI_hi'].values - fvals
yp     = list(range(len(fnames)))
ax4.barh(yp, fvals, color=TEAL, edgecolor='white',
         xerr=[f_elo, f_ehi], capsize=5,
         error_kw={'elinewidth': 1.5}, height=0.5)
ax4.set_yticks(yp); ax4.set_yticklabels(fnames)
ax4.set_xlabel('Feature Importance (%) with 95% Bootstrap CI')
ax4.set_title('Figure 4: HC-WCI Feature Importance (Model D)\nGini Impurity, 200-iteration Bootstrap', fontsize=11)
for i, (v, lo, hi) in enumerate(zip(fvals, f_elo, f_ehi)):
    ax4.text(v+hi+0.5, i, f'{v:.1f}%', va='center', fontsize=10)
plt.tight_layout()
save_fig(fig4, 'Figure4_Feature_Importance')

print(f'\nAll 4 figures saved as PNG and PDF to: {FIGURES_DIR}')


  GENERATE PUBLICATION FIGURES
[22:42:26] INFO     Figure saved: Figure1_Model_Performance
[22:42:26] INFO     Figure saved: Figure2_Rp_Validation
[22:42:27] INFO     Figure saved: Figure3_SCF_Credit_Exclusion
[22:42:27] INFO     Figure saved: Figure4_Feature_Importance

All 4 figures saved as PNG and PDF to: C:\Users\Amirh\OneDrive - Wright State University\Research\HC_WCI_Research\05_Analysis\figures


In [25]:
# ================================================================
# CELL 12 — SAVE MODELS AND FINAL REPORT
# ================================================================

section('SAVE MODELS AND FINAL REPORT')

for fname, model in [
    ('Model_A_RF_HCWCIwithRp.pkl',   rf_A),
    ('Model_B_RF_IncomeOnly.pkl',     rf_B),
    ('Model_C_LR_IncomeOnly.pkl',     lr_C),
    ('Model_D_RF_HCWCInoRp.pkl',     rf_D),
]:
    with open(os.path.join(MODELS_DIR, fname), 'wb') as f:
        pickle.dump(model, f)
    log(f'Model saved: {fname}')

if HAS_SHAP:
    print('Computing SHAP values for Model D...')
    expl  = shap.TreeExplainer(rf_D)
    shaps = expl.shap_values(XD_te)
    fig_s, _ = plt.subplots(figsize=(7, 4))
    shap.summary_plot(shaps, XD_te, plot_type='bar', show=False)
    plt.tight_layout()
    save_fig(fig_s, 'FigureS1_SHAP_ModelD')
    log('SHAP analysis saved.')
else:
    log('SHAP skipped (pip install shap).', level='NOTE')

section('ANALYSIS COMPLETE')
r2_C_f = results_df.loc[2, 'R2_pct']
r2_D_f = results_df.loc[3, 'R2_pct']
print(f"""
FINAL RESULTS SUMMARY
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

PART 1 — ACS ML (Primary Analysis)
  ACS cohorts       : {len(acs):,}  |  10 countries  |  51 states
  Model A R2        : {results_df.loc[0,'R2_pct']:.2f}%  HC-WCI RF with Rp (weighted)
  Model B R2        : {results_df.loc[1,'R2_pct']:.2f}%  Income RF
  Model C R2        : {results_df.loc[2,'R2_pct']:.2f}%  Income Linear  <- original baseline
  Model D R2        : {results_df.loc[3,'R2_pct']:.2f}%  HC-WCI RF no Rp  <- HEADLINE
  D minus C         : {r2_D_f - r2_C_f:.2f} percentage points

PART 2 — AHS Validation (JHE Editor Response)
  AHS units         : {len(ahs):,}
  Q4 vs Q1          : {q4_rate:.2f}% vs {q1_rate:.2f}% late payment
  Risk reduction    : {reduction:.1f}%
  Chi-square p      : {p_chi2:.6f}
  Mann-Whitney p    : {mwu_p:.6f}

PART 3 — SCF Credit Exclusion (Policy Argument)
  SCF families      : {len(scf):,}
  Overall excl rate : 17.1%
  Tier 2 vs Tier 5  : {ce_edu['Rate_Pct'][2]:.1f}% vs {ce_edu['Rate_Pct'][5]:.1f}%
  Chi-square p      : {p_ce:.6f}

OUTPUT LOCATIONS
  Tables  : {TABLES_DIR}
  Figures : {FIGURES_DIR}
  Models  : {MODELS_DIR}
""")

with open(os.path.join(OUT_DIR, 'Analysis_Report.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(LOG))
log('Analysis_Report.txt saved.')


  SAVE MODELS AND FINAL REPORT
[22:42:36] INFO     Model saved: Model_A_RF_HCWCIwithRp.pkl
[22:42:37] INFO     Model saved: Model_B_RF_IncomeOnly.pkl
[22:42:37] INFO     Model saved: Model_C_LR_IncomeOnly.pkl
[22:42:37] INFO     Model saved: Model_D_RF_HCWCInoRp.pkl
Computing SHAP values for Model D...
[22:42:51] INFO     Figure saved: FigureS1_SHAP_ModelD
[22:42:51] INFO     SHAP analysis saved.

  ANALYSIS COMPLETE

FINAL RESULTS SUMMARY
Generated: 2026-05-11 22:42:51

PART 1 — ACS ML (Primary Analysis)
  ACS cohorts       : 30,397  |  10 countries  |  51 states
  Model A R2        : -3.53%  HC-WCI RF with Rp (weighted)
  Model B R2        : -22.69%  Income RF
  Model C R2        : 5.59%  Income Linear  <- original baseline
  Model D R2        : 14.02%  HC-WCI RF no Rp  <- HEADLINE
  D minus C         : 8.43 percentage points

PART 2 — AHS Validation (JHE Editor Response)
  AHS units         : 55,669
  Q4 vs Q1          : 4.65% vs 8.64% late payment
  Risk reduction    : 46.2%
  Chi